In [1]:
from pathlib import Path
import subprocess

# Lokasi repository di Google Colab
REPO_DIR = Path(
    "/content/analisis-pola-lapangan-sepak-bola-jatim"
)

# Clone repository jika belum tersedia
if not REPO_DIR.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/AniqFarichatusZahwa/"
            "analisis-pola-lapangan-sepak-bola-jatim.git",
            str(REPO_DIR)
        ],
        check=True
    )

# Tentukan lokasi folder CSV
DATA_DIR = REPO_DIR / "data" / "raw"

print("Folder data:", DATA_DIR)
print("Folder ditemukan:", DATA_DIR.exists())

print("\nFile yang tersedia:")
if DATA_DIR.exists():
    for file in DATA_DIR.iterdir():
        print("-", file.name)

Folder data: /content/analisis-pola-lapangan-sepak-bola-jatim/data/raw
Folder ditemukan: True

File yang tersedia:
- data_lapangan.csv
- data_event_liga1.csv
- data_event_soeratin.csv
- data_event_aff.csv


In [2]:
from google.colab import files
from pathlib import Path
import pandas as pd

# Unggah file hasil perapian format koordinat yang sudah kamu unduh.
# Pilih: data_lapangan_format_koordinat_desimal.csv
uploaded_files = files.upload()

expected_filename = "data_lapangan_format_koordinat_desimal.csv"
if expected_filename not in uploaded_files:
    raise FileNotFoundError(
        f"File {expected_filename} belum diunggah. "
        "Unggah file CSV hasil perapian koordinat, lalu jalankan ulang sel ini."
    )

DATA_DIR = Path(
    "/content/analisis-pola-lapangan-sepak-bola-jatim/data/raw"
)
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Simpan sebagai file baru agar CSV asli tetap tersedia sebagai cadangan.
target_path = DATA_DIR / "data_lapangan_final.csv"
target_path.write_bytes(uploaded_files[expected_filename])

# Validasi struktur dan koordinat sebelum pipeline dijalankan.
cek = pd.read_csv(
    target_path,
    sep=";",
    encoding="utf-8-sig"
)

kolom_wajib = {
    "nama_venue", "latitude", "longitude",
    "jumlah_event", "luas_m2", "kapasitas_penonton"
}
kolom_hilang = kolom_wajib - set(cek.columns)
if kolom_hilang:
    raise ValueError(f"Kolom wajib tidak ada: {sorted(kolom_hilang)}")

for kolom in ["latitude", "longitude"]:
    cek[kolom] = pd.to_numeric(cek[kolom], errors="coerce")

if cek[["latitude", "longitude"]].isna().any().any():
    raise ValueError(
        "Ada koordinat kosong/tidak terbaca. "
        "Periksa kembali file CSV koordinat final."
    )

valid = (
    cek["latitude"].between(-90, 90)
    & cek["longitude"].between(-180, 180)
)
if not valid.all():
    display(cek.loc[~valid, ["nama_venue", "latitude", "longitude"]])
    raise ValueError("Ada koordinat di luar rentang yang valid.")

print("Dataset fasilitas final siap digunakan.")
print("Jumlah venue:", len(cek))
print("Koordinat lengkap:", int(cek[["latitude", "longitude"]].notna().all(axis=1).sum()))
print("File disimpan sebagai:", target_path)


Saving data_lapangan_format_koordinat_desimal.csv to data_lapangan_format_koordinat_desimal.csv
Dataset fasilitas final siap digunakan.
Jumlah venue: 67
Koordinat lengkap: 67
File disimpan sebagai: /content/analisis-pola-lapangan-sepak-bola-jatim/data/raw/data_lapangan_final.csv


In [3]:

from pathlib import Path
import pandas as pd

# Lokasi folder CSV di Google Colab
DATA_DIR = Path(
    "/content/analisis-pola-lapangan-sepak-bola-jatim/data/raw"
)

files = {
    "Soeratin": DATA_DIR / "data_event_soeratin.csv",
    "Liga 1": DATA_DIR / "data_event_liga1.csv",
    "AFF U-19": DATA_DIR / "data_event_aff.csv",
    "Fasilitas": DATA_DIR / "data_lapangan_final.csv",
}

# Menyimpan data yang berhasil dibaca
datasets = {}

for nama, path in files.items():
    print("\n" + "=" * 60)
    print(f"DATASET: {nama}")
    print("=" * 60)

    if not path.exists():
        print(f"File tidak ditemukan: {path}")
        continue

    # Penting: gunakan separator titik koma (;)
    # dtype=str mempertahankan format nilai asli saat audit
    df = pd.read_csv(
        path,
        sep=";",
        encoding="utf-8-sig",
        dtype=str
    )

    datasets[nama] = df

    print(f"Jumlah baris dan kolom: {df.shape}")
    print(
        "Jumlah baris kosong seluruhnya:",
        df.isna().all(axis=1).sum()
    )
    print(
        "Jumlah baris duplikat persis:",
        df.duplicated().sum()
    )

    print("\nNama kolom:")
    print(df.columns.tolist())

    print("\nNilai kosong per kolom:")
    print(
        df.isna().sum().sort_values(ascending=False)
    )

    print("\nContoh 5 baris:")
    display(df.head())



DATASET: Soeratin
Jumlah baris dan kolom: (824, 9)
Jumlah baris kosong seluruhnya: 0
Jumlah baris duplikat persis: 0

Nama kolom:
['id_event', 'nama_event', 'tanggal_mulai', 'tanggal_selesai', 'nama_venue', 'level_event', 'peserta_tim', 'hasil_skor', 'sumber']

Nilai kosong per kolom:
level_event        577
hasil_skor         160
nama_venue          20
id_event             0
nama_event           0
tanggal_selesai      0
tanggal_mulai        0
peserta_tim          0
sumber               0
dtype: int64

Contoh 5 baris:


,id_event,nama_event,tanggal_mulai,tanggal_selesai,nama_venue,level_event,peserta_tim,hasil_skor,sumber
0,SOERATIN-U17-001,Piala Soeratin U-17,03/12/2023,03/12/2023,Stadion Canda Bhirawa,NaN,Deltras Sidoarjo U-17 VS Persedikab Kediri U-17,00:04,https://pssijatim.com/page-piala-soeratin-u-17
1,SOERATIN-U17-002,Piala Soeratin U-17,03/12/2023,03/12/2023,Stadion Canda Bhirawa,NaN,Sinar Harapan U-17 VS Perseta Tulungagung U-17,03:02,https://pssijatim.com/page-piala-soeratin-u-17
2,SOERATIN-U17-003,Piala Soeratin U-17,01/12/2023,01/12/2023,Stadion Canda Bhirawa,NaN,Perseta Tulungagung U-17 VS Persedikab Kediri ...,01:03,https://pssijatim.com/page-piala-soeratin-u-17
3,SOERATIN-U17-004,Piala Soeratin U-17,01/12/2023,01/12/2023,Stadion Canda Bhirawa,NaN,Deltras Sidoarjo U-17 VS Sinar Harapan U-17,01:00,https://pssijatim.com/page-piala-soeratin-u-17
4,SOERATIN-U17-005,Piala Soeratin U-17,27/11/2023,27/11/2023,Stadion Soeprijadi Blitar,NaN,PSBR Kota Blitar U-17 VS NZR Sumbersari U-17,02:01,https://pssijatim.com/page-piala-soeratin-u-17



DATASET: Liga 1
Jumlah baris dan kolom: (875, 9)
Jumlah baris kosong seluruhnya: 841
Jumlah baris duplikat persis: 840

Nama kolom:
['id_event', 'nama_event', 'tanggal_mulai', 'tanggal_selesai', 'nama_venue', 'level_event', 'peserta_tim', 'hasil_skor', 'sumber']

Nilai kosong per kolom:
id_event           841
nama_event         841
tanggal_mulai      841
tanggal_selesai    841
nama_venue         841
level_event        841
peserta_tim        841
hasil_skor         841
sumber             841
dtype: int64

Contoh 5 baris:


,id_event,nama_event,tanggal_mulai,tanggal_selesai,nama_venue,level_event,peserta_tim,hasil_skor,sumber
0,LIGA1-2023-001,Liga 1 2023/2024,08/07/2023,08/07/2023,Gelora Bung Tomo,Nasional,PERSEBAYA SURABAYA vs PS BARITO PUTERA,01:01,https://www.ligaindonesiabaru.com/clubs/single...
1,LIGA1-2023-002,Liga 1 2023/2024,23/07/2023,23/07/2023,Gelora Bung Tomo,Nasional,PERSEBAYA SURABAYA vs RANS Nusantara FC,02:02,https://www.ligaindonesiabaru.com/clubs/single...
2,LIGA1-2023-003,Liga 1 2023/2024,04/08/2023,04/08/2023,Gelora Bung Tomo,Nasional,PERSEBAYA SURABAYA vs PERSIKABO 1973,01:02,https://www.ligaindonesiabaru.com/clubs/single...
3,LIGA1-2023-004,Liga 1 2023/2024,12/08/2023,12/08/2023,Gelora Bung Tomo,Nasional,PERSEBAYA SURABAYA vs PERSITA,01:00,https://www.ligaindonesiabaru.com/clubs/single...
4,LIGA1-2023-005,Liga 1 2023/2024,18/08/2023,18/08/2023,Gelora Bung Tomo,Nasional,PERSEBAYA SURABAYA vs PSM MAKASSAR,01:00,https://www.ligaindonesiabaru.com/clubs/single...



DATASET: AFF U-19
Jumlah baris dan kolom: (17, 9)
Jumlah baris kosong seluruhnya: 0
Jumlah baris duplikat persis: 0

Nama kolom:
['id_event', 'nama_event', 'tanggal_mulai', 'tanggal_selesai', 'nama_venue', 'level_event', 'peserta_tim', 'hasil_skor', 'sumber']

Nilai kosong per kolom:
hasil_skor         1
id_event           0
nama_event         0
tanggal_selesai    0
tanggal_mulai      0
nama_venue         0
level_event        0
peserta_tim        0
sumber             0
dtype: int64

Contoh 5 baris:


,id_event,nama_event,tanggal_mulai,tanggal_selesai,nama_venue,level_event,peserta_tim,hasil_skor,sumber
0,AFF19-001,Piala AFF U-19 2024,18/07/2024,18/07/2024,Gelora 10 Nopember,Internasional,Laos vs Australia,00:06,https://www.bola.net/tim_nasional/hasil-lengka...
1,AFF19-002,Piala AFF U-19 2024,18/07/2024,18/07/2024,Gelora 10 Nopember,Internasional,Vietnam vs Myanmar,01:01,https://www.bola.net/tim_nasional/hasil-lengka...
2,AFF19-003,Piala AFF U-19 2024,19/07/2024,19/07/2024,Gelora 10 Nopember,Internasional,Malaysia vs Brunei,11:00,https://www.bola.net/tim_nasional/hasil-lengka...
3,AFF19-004,Piala AFF U-19 2024,19/07/2024,19/07/2024,Gelora 10 Nopember,Internasional,Thailand vs Singapura,02:01,https://www.bola.net/tim_nasional/hasil-lengka...
4,AFF19-005,Piala AFF U-19 2024,21/07/2024,21/07/2024,Gelora 10 Nopember,Internasional,Australia vs Vietnam,06:02,https://www.bola.net/tim_nasional/hasil-lengka...



DATASET: Fasilitas
Jumlah baris dan kolom: (67, 12)
Jumlah baris kosong seluruhnya: 0
Jumlah baris duplikat persis: 0

Nama kolom:
['nama_venue', 'jumlah_event', 'kecamatan', 'kelurahan', 'nama_fasilitas_olahraga', 'alamatlokasi', 'luas_m2', 'kapasitas_penonton', 'latitude', 'longitude', 'link_google_maps', 'link_titik_koordinat']

Nilai kosong per kolom:
luas_m2                    43
kapasitas_penonton         40
kelurahan                  21
kecamatan                  14
alamatlokasi               11
jumlah_event                0
nama_venue                  0
nama_fasilitas_olahraga     0
latitude                    0
longitude                   0
link_google_maps            0
link_titik_koordinat        0
dtype: int64

Contoh 5 baris:


,nama_venue,jumlah_event,kecamatan,kelurahan,nama_fasilitas_olahraga,alamatlokasi,luas_m2,kapasitas_penonton,latitude,longitude,link_google_maps,link_titik_koordinat
0,Stadion Canda Bhirawa,48,Pare,Tulungrejo,Stadion Canda Bhirawa,"Jl. PB Sudirman, Puhrejo, Tulungrejo, Kec. Par...",NaN,3.000,-7.7626776,112.1848437,https://www.google.com/maps/search/?api=1&quer...,https://www.google.com/maps/search/?api=1&quer...
1,Gelora Bung Tomo,40,Benowo,Benowo,Gelora Bung Tomo,"Jl. Jawar, Kecamatan Benowo, Kota Surabaya",NaN,45.000,-7.2231559,112.6227833,https://www.google.com/maps/search/?api=1&quer...,https://www.google.com/maps/search/?api=1&quer...
2,Stadion Surajaya Lamongan,40,Deket,Deket Kulon,Stadion Surajaya Lamongan,"Jl. Raya Gresik–Babat, Deket Kulon, Kec. Deket...","53.144,00",11.214,-7.1133922,112.4294670,https://www.google.com/maps/search/?api=1&quer...,https://www.google.com/maps/search/?api=1&quer...
3,Stadion Soeprijadi Blitar,37,Kepanjenkidul,Kepanjen Lor,Stadion Soeprijadi Blitar,"Jl. Kelud No.125, Kepanjen Lor, Kec. Kepanjenk...","15.358,00",15.000,-8.0929997,112.1681169,https://www.google.com/maps/search/?api=1&quer...,https://www.google.com/maps/search/?api=1&quer...
4,Jember Sport Garden,36,Ajung,Ajung Kulon,Jember Sport Garden,"Kompleks Jember Sport Garden, Jl. M. Yamin, Ke...","97.000,00",20.000,-8.2101539,113.6648642,https://www.google.com/maps/search/?api=1&quer...,https://www.google.com/maps/search/?api=1&quer...


In [4]:
from pathlib import Path
import pandas as pd

# Lokasi dataset mentah
BASE_DIR = Path(
    "/content/analisis-pola-lapangan-sepak-bola-jatim"
)
RAW_DIR = BASE_DIR / "data" / "raw"

# Membaca data tanpa mengubah file mentah
def baca_csv(nama_file):
    df = pd.read_csv(
        RAW_DIR / nama_file,
        sep=";",
        encoding="utf-8-sig",
        dtype=str
    )

    # Spasi kosong dianggap sebagai nilai kosong
    df = df.replace(r"^\s*$", pd.NA, regex=True)

    # Buang baris yang seluruh kolomnya kosong,
    # hanya untuk pemeriksaan di memori
    df = df.dropna(how="all").copy()

    # Hilangkan spasi di awal/akhir teks
    for kolom in df.select_dtypes(include="object").columns:
        df[kolom] = df[kolom].str.strip()

    return df


soeratin = baca_csv("data_event_soeratin.csv")
liga1 = baca_csv("data_event_liga1.csv")
aff = baca_csv("data_event_aff.csv")
fasilitas = baca_csv("data_lapangan_final.csv")


# --------------------------------------------------
# A. AUDIT SKOR DAN ID PERTANDINGAN
# --------------------------------------------------
def audit_event(nama, df):
    print("\n" + "=" * 65)
    print(f"AUDIT PERTANDINGAN: {nama}")
    print("=" * 65)

    print("Jumlah pertandingan:", len(df))

    # Periksa duplikasi ID
    print(
        "ID pertandingan duplikat:",
        df["id_event"].duplicated().sum()
    )

    # Ringkasan nilai kosong
    print("\nNilai kosong:")
    print(
        df.isna().sum().sort_values(ascending=False)
    )

    # Distribusi format skor
    print("\n15 nilai skor yang paling sering muncul:")
    print(
        df["hasil_skor"]
        .value_counts(dropna=False)
        .head(15)
    )

    # Baris yang skornya kosong
    print("\nContoh pertandingan tanpa skor:")
    display(
        df.loc[
            df["hasil_skor"].isna(),
            ["id_event", "tanggal_mulai",
             "nama_venue", "peserta_tim", "hasil_skor"]
        ].head(10)
    )

    # Pertandingan tanpa venue
    print("\nPertandingan tanpa venue:")
    display(
        df.loc[
            df["nama_venue"].isna(),
            ["id_event", "tanggal_mulai",
             "peserta_tim", "hasil_skor"]
        ].head(30)
    )

    # Nilai level/babak yang tersedia
    if "level_event" in df.columns:
        print("\nDistribusi level_event:")
        print(
            df["level_event"]
            .value_counts(dropna=False)
            .head(20)
        )


audit_event("Soeratin", soeratin)
audit_event("Liga 1", liga1)
audit_event("AFF U-19", aff)


# --------------------------------------------------
# B. AUDIT DATA FASILITAS
# --------------------------------------------------
print("\n" + "=" * 65)
print("AUDIT DATA FASILITAS")
print("=" * 65)

print("Jumlah venue:", len(fasilitas))

print("\nNilai kosong per kolom:")
print(
    fasilitas.isna().sum().sort_values(ascending=False)
)

# Periksa apakah ada nama venue yang benar-benar berulang
print(
    "\nJumlah nama venue duplikat:",
    fasilitas["nama_venue"].duplicated().sum()
)

print("\nContoh nilai luas:")
print(fasilitas["luas_m2"].dropna().head(15).tolist())

print("\nContoh nilai kapasitas:")
print(
    fasilitas["kapasitas_penonton"]
    .dropna()
    .head(15)
    .tolist()
)

# Tampilkan venue yang belum lengkap
kolom_penting = [
    "nama_venue",
    "kecamatan",
    "kelurahan",
    "alamatlokasi",
    "luas_m2",
    "kapasitas_penonton"
]

print("\nContoh venue yang belum lengkap:")
display(
    fasilitas.loc[
        fasilitas[kolom_penting].isna().any(axis=1),
        kolom_penting
    ].head(20)
)


AUDIT PERTANDINGAN: Soeratin
Jumlah pertandingan: 824
ID pertandingan duplikat: 0

Nilai kosong:
level_event        577
hasil_skor         160
nama_venue          20
id_event             0
nama_event           0
tanggal_selesai      0
tanggal_mulai        0
peserta_tim          0
sumber               0
dtype: int64

15 nilai skor yang paling sering muncul:
hasil_skor
NaN      160
00:01     63
01:01     60
01:00     57
02:01     47
00:00     43
00:02     41
02:00     39
01:02     36
00:03     25
03:01     24
04:00     21
02:02     19
03:00     18
02:03     17
Name: count, dtype: int64

Contoh pertandingan tanpa skor:


,id_event,tanggal_mulai,nama_venue,peserta_tim,hasil_skor
61,SOERATIN-U17-062,11/11/2023,Lapangan CJI Samsung Pasuruan,Gresik United U-17 VS Perssu Sumenep U-17,NaN
85,SOERATIN-U17-086,09/11/2023,Lapangan CJI Samsung Pasuruan,Gresik United U-17 VS Persekap Pasuruan U-17,NaN
109,SOERATIN-U17-110,08/11/2023,Lapangan CJI Samsung Pasuruan,Bhayangkara FC U-17 VS Gresik United U-17,NaN
200,SOERATIN-U17-201,20/11/2024,"Stadion Mini Dolok Gede, Bojonegoro",Cahaya Muda VS Perspa Pacitan,NaN
227,SOERATIN-U17-228,18/11/2024,"Stadion Mini Dolok Gede, Bojonegoro",Cahaya Muda VS Persibo Bojonegoro,NaN
257,SOERATIN-U17-258,16/11/2024,"Stadion Mini Dolok Gede, Bojonegoro",Persinga Ngawi VS Cahaya Muda,NaN
545,SOERATIN-U13-001,08/01/2024,Lapangan Secaba Jember,PS HW Babat VS Turangga FC,NaN
546,SOERATIN-U13-002,08/01/2024,"Lapangan Yonif 509, Jember",Bangkalan Soccer Academy VS Putra Manggala,NaN
547,SOERATIN-U13-003,08/01/2024,"Lapangan Brigif Bawah, Jember",Phoenix Mojokerto FC VS Argent Genteng,NaN
548,SOERATIN-U13-004,08/01/2024,"Lapangan Brigif Atas, Jember",Football Class VS Putra Jaya Blitar,NaN



Pertandingan tanpa venue:


,id_event,tanggal_mulai,peserta_tim,hasil_skor
600,SOERATIN-U13-056,06/08/2025,Persebaya Surabaya VS SSB Airlangga Kab. Kediri,NaN
601,SOERATIN-U13-057,06/08/2025,Persema Malang VS Deltras Sidoarjo,NaN
602,SOERATIN-U13-058,06/08/2025,SSB Laga Sota Pamekasan VS Medi Soccer School ...,NaN
603,SOERATIN-U13-059,06/08/2025,MTM Tumpang Kab. Blitar VS Putra Garuda Lumajang,NaN
604,SOERATIN-U13-060,04/08/2025,Medi Soccer School Tulungagung VS SSB Sindogre...,NaN
605,SOERATIN-U13-061,04/08/2025,Persela Lamongan VS SSB Airlangga Kab. Kediri,NaN
606,SOERATIN-U13-062,04/08/2025,Ricky Nelson Academy VS Putra Garuda Lumajang,NaN
607,SOERATIN-U13-063,04/08/2025,Deltras Sidoarjo VS Biting Putra Jaya Jember,NaN
608,SOERATIN-U13-064,04/08/2025,Narayana Lamongan VS SSB Laga Sota Pamekasan,NaN
609,SOERATIN-U13-065,04/08/2025,Persebaya Surabaya VS Balenos Nganjuk,NaN



Distribusi level_event:
level_event
NaN              577
Grup H            25
Grup D            24
Grup G            24
Grup F            24
Grup C            24
Grup E            24
Grup B            21
Grup A            21
Grup I            15
20 Besar          10
16 Besar           8
Semifinal          6
8 Besar            4
Babak 6 Besar      4
10 Besar           4
Final Juara 1      3
Final Juara 3      3
Grup J             3
Name: count, dtype: int64

AUDIT PERTANDINGAN: Liga 1
Jumlah pertandingan: 34
ID pertandingan duplikat: 0

Nilai kosong:
id_event           0
nama_event         0
tanggal_mulai      0
tanggal_selesai    0
nama_venue         0
level_event        0
peserta_tim        0
hasil_skor         0
sumber             0
dtype: int64

15 nilai skor yang paling sering muncul:
hasil_skor
01:01    8
02:01    7
01:00    6
00:02    2
00:00    2
04:01    2
01:02    1
02:02    1
03:01    1
00:03    1
02:03    1
03:02    1
01:03    1
Name: count, dtype: int64

Contoh pertandinga

...

,id_event,tanggal_mulai,nama_venue,peserta_tim,hasil_skor
12,AFF19-013,25/07/2024,Gelora 10 Nopember,Singapura vs Brunei,NaN



Pertandingan tanpa venue:


,id_event,tanggal_mulai,peserta_tim,hasil_skor



Distribusi level_event:
level_event
Internasional    17
Name: count, dtype: int64

AUDIT DATA FASILITAS
Jumlah venue: 67

Nilai kosong per kolom:
luas_m2                    43
kapasitas_penonton         40
kelurahan                  21
kecamatan                  14
alamatlokasi               11
jumlah_event                0
nama_venue                  0
nama_fasilitas_olahraga     0
latitude                    0
longitude                   0
link_google_maps            0
link_titik_koordinat        0
dtype: int64

Jumlah nama venue duplikat: 0

Contoh nilai luas:
['53.144,00', '15.358,00', '97.000,00', '27.100,00', '30.100,00', '9.084,00', '20.000,00', '6.240,00', '33.045,00', '14.681,00', '14.737,00', '39.000,00', '10.800,00', '40.000,00', '7.140,00']

Contoh nilai kapasitas:
['3.000', '45.000', '11.214', '15.000', '20.000', '10.000', '500', '9.000', '15.000', '5.000', '8.620', '10.000', '10.000', '35.000', '12.000']

Contoh venue yang belum lengkap:


,nama_venue,kecamatan,kelurahan,alamatlokasi,luas_m2,kapasitas_penonton
0,Stadion Canda Bhirawa,Pare,Tulungrejo,"Jl. PB Sudirman, Puhrejo, Tulungrejo, Kec. Par...",NaN,3.000
1,Gelora Bung Tomo,Benowo,Benowo,"Jl. Jawar, Kecamatan Benowo, Kota Surabaya",NaN,45.000
5,Lapangan Pagu Kediri,NaN,NaN,NaN,NaN,NaN
6,Stadion Jenggolo Sidoarjo,Buduran,Siwalanpanji,"Jl. Jenggolo, Siwalanpanji, Kec. Buduran, Kab....","27.100,00",NaN
8,"Lapangan Gajah Mada, Lamongan",Lamongan,Sidoharjo,"Jl. Sumargo, Kel. Sidoharjo, Kec. Lamongan, Ka...","9.084,00",NaN
9,Lapangan Ngeronggo Kediri,Kota,Ngronggo,"Jl. Perintis Kemerdekaan No.175, Ngronggo, Kot...","20.000,00",NaN
11,Lapangan Polda Jatim,Gayungan,Ketintang,"Jl. Ahmad Yani No.116, Ketintang, Kec. Gayunga...",NaN,NaN
13,Lapangan Ngino Kediri,NaN,NaN,NaN,NaN,NaN
14,"Stadion Mini Dolok Gede, Bojonegoro",Tambakrejo,Dolokgede,"Desa Dolokgede, Kec. Tambakrejo, Kab. Bojonegoro","14.681,00",NaN
15,Stadion Semeru Lumajang,Lumajang,Kepuharjo,"Jl. Gajah Mada No.1, Kepuharjo, Kec. Lumajang,...","14.737,00",NaN


In [5]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path(
    "/content/analisis-pola-lapangan-sepak-bola-jatim"
)
RAW_DIR = BASE_DIR / "data" / "raw"
OUT_DIR = BASE_DIR / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

EVENT_FILES = {
    "Soeratin": "data_event_soeratin.csv",
    "Liga 1": "data_event_liga1.csv",
    "AFF U-19": "data_event_aff.csv",
}

# Membaca CSV dan membersihkan baris kosong sepenuhnya
def read_clean_csv(filename):
    df = pd.read_csv(
        RAW_DIR / filename,
        sep=";",
        encoding="utf-8-sig",
        dtype=str
    )

    df = df.replace(r"^\s*$", pd.NA, regex=True)

    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].str.strip()

    return df.dropna(how="all").copy()


# Mengubah skor seperti "00:04" menjadi dua angka.
# Skor asli tetap disimpan dan tidak ditimpa.
def prepare_event(df, name):
    score_text = df["hasil_skor"].astype("string")

    parts = score_text.str.extract(
        r"^\s*(\d{1,2})\s*[:\-–]\s*(\d{1,2})\s*$"
    )

    df["skor_tim_1"] = pd.to_numeric(
        parts[0], errors="coerce"
    ).astype("Int64")

    df["skor_tim_2"] = pd.to_numeric(
        parts[1], errors="coerce"
    ).astype("Int64")

    valid_score = (
        df["skor_tim_1"].notna()
        & df["skor_tim_2"].notna()
    )

    df["hasil_skor_normalisasi"] = pd.Series(
        pd.NA, index=df.index, dtype="string"
    )

    df.loc[valid_score, "hasil_skor_normalisasi"] = (
        df.loc[valid_score, "skor_tim_1"].astype(str)
        + "-"
        + df.loc[valid_score, "skor_tim_2"].astype(str)
    )

    # Standarkan tanggal dalam kolom baru, bukan menimpa tanggal asli
    for col in ["tanggal_mulai", "tanggal_selesai"]:
        parsed = pd.to_datetime(
            df[col],
            format="%d/%m/%Y",
            errors="coerce"
        )

        failed = df[col].notna() & parsed.isna()

        df[f"{col}_iso"] = parsed.dt.strftime("%Y-%m-%d")

        if failed.sum():
            print(
                f"Peringatan {name}: {failed.sum()} "
                f"nilai {col} gagal dikonversi."
            )

    # Catat hasil validasi skor
    unparsed = df["hasil_skor"].notna() & ~valid_score

    print(f"\n{name}")
    print("Jumlah pertandingan:", len(df))
    print("Skor berhasil dibaca:", int(valid_score.sum()))
    print("Skor kosong:", int(df["hasil_skor"].isna().sum()))
    print("Skor tidak sesuai pola:", int(unparsed.sum()))
    print("Venue kosong:", int(df["nama_venue"].isna().sum()))
    print("Level event kosong:", int(df["level_event"].isna().sum()))

    if unparsed.sum():
        print("Contoh skor yang perlu diperiksa:")
        print(df.loc[unparsed, "hasil_skor"].unique()[:10])

    return df


# A. Siapkan tiga dataset pertandingan
for name, filename in EVENT_FILES.items():
    df = read_clean_csv(filename)
    df = prepare_event(df, name)

    output = OUT_DIR / filename.replace(".csv", "_clean.csv")

    df.to_csv(
        output,
        sep=";",
        index=False,
        encoding="utf-8-sig"
    )

    print("Disimpan:", output)


# B. Konversi angka dengan format Indonesia
def parse_indonesian_number(value):
    if pd.isna(value):
        return pd.NA

    s = str(value).strip()

    if s.lower() in {"", "nan", "none", "na", "-"}:
        return pd.NA

    # Contoh: 53.144,00 -> 53144.00
    if "." in s and "," in s:
        s = s.replace(".", "").replace(",", ".")

    # Contoh: 53,5 -> 53.5
    elif "," in s:
        s = s.replace(",", ".")

    # Contoh: 45.000 -> 45000
    elif "." in s:
        chunks = s.split(".")
        if (
            len(chunks[-1]) == 3
            and all(chunk.isdigit() for chunk in chunks)
        ):
            s = "".join(chunks)

    try:
        return float(s)
    except ValueError:
        return pd.NA


# C. Siapkan dataset fasilitas
fasilitas = read_clean_csv("data_lapangan_final.csv")

for col in ["luas_m2", "kapasitas_penonton", "jumlah_event"]:
    fasilitas[f"{col}_asli"] = fasilitas[col]
    fasilitas[col] = fasilitas[col].map(
        parse_indonesian_number
    ).astype("Float64")

# Latitude dan longitude pada file final sudah memakai
# format desimal bertitik, sehingga dikonversi langsung ke numerik.
for col in ["latitude", "longitude"]:
    if col not in fasilitas.columns:
        raise KeyError(
            f"Kolom {col} tidak ditemukan pada data fasilitas final."
        )

    fasilitas[col] = pd.to_numeric(
        fasilitas[col], errors="coerce"
    ).astype("Float64")

# Validasi kelengkapan dan rentang koordinat sebelum integrasi
if fasilitas[["latitude", "longitude"]].isna().any().any():
    baris_kosong = fasilitas.loc[
        fasilitas[["latitude", "longitude"]].isna().any(axis=1),
        ["nama_venue", "latitude", "longitude"]
    ]
    display(baris_kosong)
    raise ValueError(
        "Masih ada venue tanpa latitude/longitude. "
        "Periksa kembali data_lapangan_final.csv."
    )

koordinat_valid = (
    fasilitas["latitude"].between(-90, 90)
    & fasilitas["longitude"].between(-180, 180)
)

if not koordinat_valid.all():
    display(
        fasilitas.loc[
            ~koordinat_valid,
            ["nama_venue", "latitude", "longitude"]
        ]
    )
    raise ValueError(
        "Ada koordinat di luar rentang latitude/longitude yang valid."
    )

print("Koordinat lengkap:", len(fasilitas))
print("Rentang latitude:",
      fasilitas["latitude"].min(), "s.d.", fasilitas["latitude"].max())
print("Rentang longitude:",
      fasilitas["longitude"].min(), "s.d.", fasilitas["longitude"].max())

# Cek apakah ada nama venue duplikat persis
print("\nFASILITAS")
print("Jumlah venue:", len(fasilitas))
print(
    "Nama venue duplikat:",
    fasilitas["nama_venue"].duplicated().sum()
)
print("\nJumlah nilai kosong setelah konversi:")
print(
    fasilitas[
        ["luas_m2", "kapasitas_penonton",
         "kecamatan", "kelurahan", "alamatlokasi"]
    ].isna().sum()
)

output = OUT_DIR / "data_lapangan_clean.csv"

fasilitas.to_csv(
    output,
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

print("\nDisimpan:", output)
print("\nPreprocessing awal selesai.")


Soeratin
Jumlah pertandingan: 824
Skor berhasil dibaca: 664
Skor kosong: 160
Skor tidak sesuai pola: 0
Venue kosong: 20
Level event kosong: 577
Disimpan: /content/analisis-pola-lapangan-sepak-bola-jatim/data/processed/data_event_soeratin_clean.csv

Liga 1
Jumlah pertandingan: 34
Skor berhasil dibaca: 34
Skor kosong: 0
Skor tidak sesuai pola: 0
Venue kosong: 0
Level event kosong: 0
Disimpan: /content/analisis-pola-lapangan-sepak-bola-jatim/data/processed/data_event_liga1_clean.csv

AFF U-19
Jumlah pertandingan: 17
Skor berhasil dibaca: 16
Skor kosong: 1
Skor tidak sesuai pola: 0
Venue kosong: 0
Level event kosong: 0
Disimpan: /content/analisis-pola-lapangan-sepak-bola-jatim/data/processed/data_event_aff_clean.csv
Koordinat lengkap: 67
Rentang latitude: -8.2934107 s.d. -7.0429502
Rentang longitude: 110.6117634 s.d. 114.3665514

FASILITAS
Jumlah venue: 67
Nama venue duplikat: 0

Jumlah nilai kosong setelah konversi:
luas_m2               43
kapasitas_penonton    40
kecamatan             

In [6]:
from pathlib import Path
import pandas as pd
import re
import unicodedata

BASE_DIR = Path(
    "/content/analisis-pola-lapangan-sepak-bola-jatim"
)
DATA_DIR = BASE_DIR / "data" / "processed"

# Membaca hasil preprocessing awal
event_files = {
    "Soeratin": "data_event_soeratin_clean.csv",
    "Liga 1": "data_event_liga1_clean.csv",
    "AFF U-19": "data_event_aff_clean.csv",
}

events = []

for kompetisi, filename in event_files.items():
    df = pd.read_csv(
        DATA_DIR / filename,
        sep=";",
        encoding="utf-8-sig",
        dtype=str
    )
    df["kompetisi_sumber"] = kompetisi
    events.append(df)

# Gabungkan data pertandingan hanya untuk proses audit
all_events = pd.concat(events, ignore_index=True)

# Baca data fasilitas
facilities = pd.read_csv(
    DATA_DIR / "data_lapangan_clean.csv",
    sep=";",
    encoding="utf-8-sig",
    dtype=str
)

# Fungsi normalisasi nama untuk membandingkan ejaan
def normalize_venue(value):
    if pd.isna(value) or not str(value).strip():
        return pd.NA

    s = unicodedata.normalize("NFKD", str(value))
    s = "".join(
        ch for ch in s
        if not unicodedata.combining(ch)
    ).lower().strip()

    # Samakan beberapa singkatan umum
    s = re.sub(r"\blap(?:angan)?\.?\b", "lapangan", s)
    s = re.sub(r"\bstad(?:ion)?\.?\b", "stadion", s)

    # Buang tanda baca dan rapikan spasi
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    return s


all_events["venue_norm"] = (
    all_events["nama_venue"].map(normalize_venue)
)

facilities["venue_norm"] = (
    facilities["nama_venue"].map(normalize_venue)
)

# Periksa apakah nama venue fasilitas unik setelah normalisasi
facility_names = facilities[
    ["nama_venue", "venue_norm"]
].dropna().drop_duplicates()

duplicate_norm = facility_names[
    facility_names.duplicated("venue_norm", keep=False)
].sort_values("venue_norm")

print("=== PEMERIKSAAN NAMA VENUE FASILITAS ===")
print("Jumlah venue fasilitas:", len(facilities))
print("Jumlah nama normalisasi yang bertabrakan:",
      duplicate_norm["venue_norm"].nunique())

if len(duplicate_norm):
    display(duplicate_norm)

# Daftar venue fasilitas untuk pencocokan
facility_set = set(facilities["venue_norm"].dropna())

# Hitung pertandingan per nama venue
venue_summary = (
    all_events.dropna(subset=["venue_norm"])
    .groupby(["nama_venue", "venue_norm"], as_index=False)
    .agg(
        jumlah_pertandingan=("id_event", "size"),
        kompetisi=("kompetisi_sumber",
                   lambda s: ", ".join(sorted(set(s))))
    )
)

venue_summary["status_pencocokan"] = (
    venue_summary["venue_norm"]
    .isin(facility_set)
    .map({True: "Cocok", False: "Belum cocok"})
)

# Ringkasan pencocokan
print("\n=== RINGKASAN PENCocOKAN ===")
print("Total baris pertandingan:", len(all_events))
print("Pertandingan tanpa nama venue:",
      all_events["nama_venue"].isna().sum())
print("Nama venue unik pada data event:",
      all_events["venue_norm"].nunique())

print("\nStatus pencocokan:")
display(
    venue_summary.groupby("status_pencocokan")
    .agg(
        jumlah_nama_venue=("nama_venue", "nunique"),
        jumlah_pertandingan=("jumlah_pertandingan", "sum")
    )
    .reset_index()
)

# Tampilkan nama venue yang belum cocok
unmatched = (
    venue_summary[
        venue_summary["status_pencocokan"] == "Belum cocok"
    ]
    .sort_values("jumlah_pertandingan", ascending=False)
)

print("\n=== VENUE EVENT YANG BELUM COCOK ===")
display(unmatched)

# Tampilkan venue fasilitas yang tidak muncul dalam event
unused_facilities = facilities[
    ~facilities["venue_norm"].isin(
        set(all_events["venue_norm"].dropna())
    )
][["nama_venue", "kecamatan", "kelurahan"]]

print("\n=== FASILITAS YANG BELUM MUNCUL DI DATA EVENT ===")
display(unused_facilities)

=== PEMERIKSAAN NAMA VENUE FASILITAS ===
Jumlah venue fasilitas: 67
Jumlah nama normalisasi yang bertabrakan: 0

=== RINGKASAN PENCocOKAN ===
Total baris pertandingan: 875
Pertandingan tanpa nama venue: 20
Nama venue unik pada data event: 67

Status pencocokan:


,status_pencocokan,jumlah_nama_venue,jumlah_pertandingan
0,Cocok,67,855



=== VENUE EVENT YANG BELUM COCOK ===


,nama_venue,venue_norm,jumlah_pertandingan,kompetisi,status_pencocokan



=== FASILITAS YANG BELUM MUNCUL DI DATA EVENT ===


,nama_venue,kecamatan,kelurahan


In [7]:
from pathlib import Path
import pandas as pd
import re
import unicodedata
import hashlib

BASE_DIR = Path(
    "/content/analisis-pola-lapangan-sepak-bola-jatim"
)
DATA_DIR = BASE_DIR / "data" / "processed"
DATA_DIR.mkdir(parents=True, exist_ok=True)


# 1. Fungsi normalisasi nama venue
def normalize_venue(value):
    if pd.isna(value) or not str(value).strip():
        return pd.NA

    s = unicodedata.normalize("NFKD", str(value))
    s = "".join(
        ch for ch in s
        if not unicodedata.combining(ch)
    ).lower().strip()

    s = re.sub(r"\blap(?:angan)?\.?\b", "lapangan", s)
    s = re.sub(r"\bstad(?:ion)?\.?\b", "stadion", s)
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    return s


# 2. Baca tiga dataset event hasil preprocessing
event_sources = {
    "Soeratin": "data_event_soeratin_clean.csv",
    "Liga 1": "data_event_liga1_clean.csv",
    "AFF U-19": "data_event_aff_clean.csv",
}

event_list = []

for kompetisi, filename in event_sources.items():
    df = pd.read_csv(
        DATA_DIR / filename,
        sep=";",
        encoding="utf-8-sig",
        dtype=str
    )

    # Simpan asal data setiap pertandingan
    df["kompetisi_sumber"] = kompetisi
    event_list.append(df)

all_events = pd.concat(
    event_list,
    ignore_index=True
)

all_events["venue_norm"] = (
    all_events["nama_venue"].map(normalize_venue)
)


# 3. Baca data fasilitas
facilities = pd.read_csv(
    DATA_DIR / "data_lapangan_clean.csv",
    sep=";",
    encoding="utf-8-sig",
    dtype=str
)

# Pastikan koordinat tetap bertipe numerik pada dimensi venue
for col in ["latitude", "longitude"]:
    if col not in facilities.columns:
        raise KeyError(
            f"Kolom {col} tidak tersedia pada data_lapangan_clean.csv. "
            "Jalankan ulang preprocessing fasilitas terlebih dahulu."
        )
    facilities[col] = pd.to_numeric(
        facilities[col], errors="coerce"
    )

if facilities[["latitude", "longitude"]].isna().any().any():
    raise ValueError(
        "Koordinat tidak lengkap pada data fasilitas yang akan diintegrasikan."
    )

facilities["venue_norm"] = (
    facilities["nama_venue"].map(normalize_venue)
)


# 4. Validasi keunikan nama fasilitas
if facilities["venue_norm"].isna().any():
    raise ValueError(
        "Ada nama venue fasilitas yang kosong. "
        "Periksa data fasilitas terlebih dahulu."
    )

if facilities["venue_norm"].duplicated().any():
    raise ValueError(
        "Ada nama venue fasilitas yang bertabrakan "
        "setelah normalisasi. Perlu pemeriksaan manual."
    )


# 5. Buat ID venue yang stabil berdasarkan nama normalisasi
def make_venue_id(normalized_name):
    digest = hashlib.sha1(
        normalized_name.encode("utf-8")
    ).hexdigest()[:10].upper()

    return f"VEN-{digest}"


facilities["venue_id"] = (
    facilities["venue_norm"].map(make_venue_id)
)

venue_map = dict(
    zip(facilities["venue_norm"], facilities["venue_id"])
)

all_events["venue_id"] = (
    all_events["venue_norm"].map(venue_map)
)


# 6. Status pencocokan venue
def matching_status(row):
    if pd.isna(row["nama_venue"]) or not str(
        row["nama_venue"]
    ).strip():
        return "Venue kosong"

    if pd.isna(row["venue_id"]):
        return "Belum cocok"

    return "Cocok"


all_events["status_pencocokan_venue"] = (
    all_events.apply(matching_status, axis=1)
)


# 7. Hitung ulang jumlah pertandingan dari data event
jumlah_event_hasil = (
    all_events.dropna(subset=["venue_id"])
    .groupby("venue_id")
    .size()
    .rename("jumlah_event_hasil_integrasi")
    .reset_index()
)


# 8. Bandingkan dengan jumlah_event pada data fasilitas
comparison = facilities[
    ["venue_id", "nama_venue", "jumlah_event"]
].copy()

comparison = comparison.rename(
    columns={
        "jumlah_event": "jumlah_event_dataset_fasilitas"
    }
)

comparison["jumlah_event_dataset_fasilitas"] = (
    pd.to_numeric(
        comparison["jumlah_event_dataset_fasilitas"],
        errors="coerce"
    )
)

comparison = comparison.merge(
    jumlah_event_hasil,
    on="venue_id",
    how="left",
    validate="one_to_one"
)

comparison["jumlah_event_hasil_integrasi"] = (
    comparison["jumlah_event_hasil_integrasi"]
    .fillna(0)
    .astype(int)
)

comparison["selisih_event"] = (
    comparison["jumlah_event_hasil_integrasi"]
    - comparison["jumlah_event_dataset_fasilitas"]
)

comparison["status_validasi"] = comparison[
    "selisih_event"
].apply(
    lambda x: (
        "Sesuai" if pd.notna(x) and x == 0
        else "Perlu diperiksa"
    )
)


# 9. Simpan tabel analitik awal
# Pertahankan semua pertandingan, termasuk yang venue-nya kosong.
fact_event = all_events.drop(
    columns=["venue_norm"],
    errors="ignore"
)

fact_event.to_csv(
    DATA_DIR / "fact_event.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

# Dimensi venue berisi atribut fasilitas, tanpa jumlah_event lama.
dim_columns = [
    "venue_id",
    "nama_venue",
    "kecamatan",
    "kelurahan",
    "nama_fasilitas_olahraga",
    "alamatlokasi",
    "luas_m2",
    "kapasitas_penonton",
    "latitude",
    "longitude",
]

dim_columns = [
    col for col in dim_columns
    if col in facilities.columns
]

dim_venue = facilities[dim_columns].copy()

dim_venue.to_csv(
    DATA_DIR / "dim_venue.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

comparison.to_csv(
    DATA_DIR / "audit_perbandingan_jumlah_event.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

# Simpan pertandingan tanpa venue untuk ditelusuri nanti
event_tanpa_venue = all_events[
    all_events["status_pencocokan_venue"] != "Cocok"
].copy()

event_tanpa_venue.to_csv(
    DATA_DIR / "event_perlu_verifikasi.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig"
)


# 10. Tampilkan hasil validasi
print("TOTAL PERTANDINGAN:", len(fact_event))
print(
    "Pertandingan dengan venue cocok:",
    (fact_event["status_pencocokan_venue"] == "Cocok").sum()
)
print(
    "Pertandingan tanpa venue / belum cocok:",
    len(event_tanpa_venue)
)
print("TOTAL VENUE:", len(dim_venue))

print("\nSTATUS PERBANDINGAN JUMLAH EVENT:")
display(
    comparison["status_validasi"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="jumlah_venue")
)

print("\nVENUE YANG JUMLAH EVENT-NYA BERBEDA:")
display(
    comparison.loc[
        comparison["status_validasi"] != "Sesuai",
        [
            "nama_venue",
            "jumlah_event_dataset_fasilitas",
            "jumlah_event_hasil_integrasi",
            "selisih_event",
            "status_validasi"
        ]
    ]
)

print("\nFile hasil:")
print(DATA_DIR / "fact_event.csv")
print(DATA_DIR / "dim_venue.csv")
print(DATA_DIR / "audit_perbandingan_jumlah_event.csv")
print(DATA_DIR / "event_perlu_verifikasi.csv")

TOTAL PERTANDINGAN: 875
Pertandingan dengan venue cocok: 855
Pertandingan tanpa venue / belum cocok: 20
TOTAL VENUE: 67

STATUS PERBANDINGAN JUMLAH EVENT:


,status,jumlah_venue
0,Sesuai,67



VENUE YANG JUMLAH EVENT-NYA BERBEDA:


,nama_venue,jumlah_event_dataset_fasilitas,jumlah_event_hasil_integrasi,selisih_event,status_validasi



File hasil:
/content/analisis-pola-lapangan-sepak-bola-jatim/data/processed/fact_event.csv
/content/analisis-pola-lapangan-sepak-bola-jatim/data/processed/dim_venue.csv
/content/analisis-pola-lapangan-sepak-bola-jatim/data/processed/audit_perbandingan_jumlah_event.csv
/content/analisis-pola-lapangan-sepak-bola-jatim/data/processed/event_perlu_verifikasi.csv


In [8]:
!pip -q install duckdb

In [9]:
from pathlib import Path
import pandas as pd
import duckdb

# --------------------------------------------------
# 1. Tentukan folder project
# --------------------------------------------------
BASE_DIR = Path(
    "/content/analisis-pola-lapangan-sepak-bola-jatim"
)

DATA_DIR = BASE_DIR / "data" / "processed"
DB_DIR = BASE_DIR / "database" / "local"

DB_DIR.mkdir(parents=True, exist_ok=True)


# --------------------------------------------------
# 2. Daftar empat bebek
# --------------------------------------------------
database_config = [
    {
        "nama": "Soeratin",
        "file": "data_event_soeratin_clean.csv",
        "database": "bebek_soeratin.duckdb",
        "tabel": "event_soeratin",
    },
    {
        "nama": "Liga 1",
        "file": "data_event_liga1_clean.csv",
        "database": "bebek_liga1.duckdb",
        "tabel": "event_liga1",
    },
    {
        "nama": "AFF U-19",
        "file": "data_event_aff_clean.csv",
        "database": "bebek_aff.duckdb",
        "tabel": "event_aff",
    },
    {
        "nama": "Fasilitas",
        "file": "data_lapangan_clean.csv",
        "database": "bebek_fasilitas.duckdb",
        "tabel": "fasilitas",
    },
]


# --------------------------------------------------
# 3. Fungsi persiapan tipe data
# --------------------------------------------------
def prepare_types(df, nama_dataset):
    df = df.copy()

    # Konversi kolom tanggal hasil standardisasi
    for col in ["tanggal_mulai_iso", "tanggal_selesai_iso"]:
        if col in df.columns:
            df[col] = pd.to_datetime(
                df[col],
                format="%Y-%m-%d",
                errors="coerce"
            ).dt.date

    # Kolom skor harus berupa bilangan bulat
    for col in ["skor_tim_1", "skor_tim_2"]:
        if col in df.columns:
            df[col] = pd.to_numeric(
                df[col], errors="coerce"
            ).astype("Int64")

    # Kolom numerik fasilitas
    if nama_dataset == "Fasilitas":
        for col in [
            "jumlah_event",
            "luas_m2",
            "kapasitas_penonton",
            "latitude",
            "longitude"
        ]:
            if col in df.columns:
                df[col] = pd.to_numeric(
                    df[col], errors="coerce"
                )

        # Jumlah event merupakan hitungan, bukan pecahan
        if "jumlah_event" in df.columns:
            if df["jumlah_event"].notna().all():
                df["jumlah_event"] = (
                    df["jumlah_event"].astype("int64")
                )

        if "kapasitas_penonton" in df.columns:
            values = df["kapasitas_penonton"].dropna()

            if (values % 1 == 0).all():
                df["kapasitas_penonton"] = (
                    df["kapasitas_penonton"].astype("Int64")
                )

    return df


# --------------------------------------------------
# 4. Impor setiap dataset ke database masing-masing
# --------------------------------------------------
hasil = []

for cfg in database_config:
    csv_path = DATA_DIR / cfg["file"]
    db_path = DB_DIR / cfg["database"]

    if not csv_path.exists():
        raise FileNotFoundError(
            f"File tidak ditemukan: {csv_path}"
        )

    # File hasil preprocessing memakai separator ;
    df = pd.read_csv(
        csv_path,
        sep=";",
        encoding="utf-8-sig"
    )

    df = prepare_types(df, cfg["nama"])

    with duckdb.connect(str(db_path)) as con:
        con.register("df_import", df)

        con.execute(
            f"""
            CREATE OR REPLACE TABLE {cfg['tabel']} AS
            SELECT * FROM df_import
            """
        )

        jumlah_baris = con.execute(
            f"SELECT COUNT(*) FROM {cfg['tabel']}"
        ).fetchone()[0]

        jumlah_kolom = len(
            con.execute(
                f"DESCRIBE {cfg['tabel']}"
            ).fetchall()
        )

        hasil.append({
            "bebek": cfg["nama"],
            "database": cfg["database"],
            "tabel": cfg["tabel"],
            "jumlah_baris": jumlah_baris,
            "jumlah_kolom": jumlah_kolom,
            "status": "Berhasil",
        })

    print(
        f"Berhasil: {cfg['nama']} | "
        f"{jumlah_baris} baris | "
        f"{db_path}"
    )

# --------------------------------------------------
# 5. Tampilkan ringkasan
# --------------------------------------------------
print("\nRINGKASAN PEMBUATAN DATABASE")
display(pd.DataFrame(hasil))

print("\nLokasi seluruh database:")
for path in sorted(DB_DIR.glob("*.duckdb")):
    print(path.name, "-", round(path.stat().st_size / 1024, 2), "KB")

Berhasil: Soeratin | 824 baris | /content/analisis-pola-lapangan-sepak-bola-jatim/database/local/bebek_soeratin.duckdb
Berhasil: Liga 1 | 34 baris | /content/analisis-pola-lapangan-sepak-bola-jatim/database/local/bebek_liga1.duckdb
Berhasil: AFF U-19 | 17 baris | /content/analisis-pola-lapangan-sepak-bola-jatim/database/local/bebek_aff.duckdb
Berhasil: Fasilitas | 67 baris | /content/analisis-pola-lapangan-sepak-bola-jatim/database/local/bebek_fasilitas.duckdb

RINGKASAN PEMBUATAN DATABASE


,bebek,database,tabel,jumlah_baris,jumlah_kolom,status
0,Soeratin,bebek_soeratin.duckdb,event_soeratin,824,14,Berhasil
1,Liga 1,bebek_liga1.duckdb,event_liga1,34,14,Berhasil
2,AFF U-19,bebek_aff.duckdb,event_aff,17,14,Berhasil
3,Fasilitas,bebek_fasilitas.duckdb,fasilitas,67,15,Berhasil



Lokasi seluruh database:
bebek_aff.duckdb - 524.0 KB
bebek_fasilitas.duckdb - 524.0 KB
bebek_liga1.duckdb - 524.0 KB
bebek_soeratin.duckdb - 524.0 KB


In [10]:
from pathlib import Path
import duckdb
import pandas as pd

# ==================================================
# 1. Lokasi project
# ==================================================
BASE_DIR = Path(
    "/content/analisis-pola-lapangan-sepak-bola-jatim"
)

LOCAL_DIR = BASE_DIR / "database" / "local"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

DUCKLAKE_DIR = BASE_DIR / "database" / "ducklake"
DATA_PATH = DUCKLAKE_DIR / "lakehouse_storage"
CATALOG_PATH = DUCKLAKE_DIR / "shared_catalog.ducklake"

DUCKLAKE_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH.mkdir(parents=True, exist_ok=True)

# Pastikan database sumber tersedia
required_databases = [
    LOCAL_DIR / "bebek_soeratin.duckdb",
    LOCAL_DIR / "bebek_liga1.duckdb",
    LOCAL_DIR / "bebek_aff.duckdb",
    LOCAL_DIR / "bebek_fasilitas.duckdb",
]

for db_path in required_databases:
    if not db_path.exists():
        raise FileNotFoundError(
            f"Database sumber tidak ditemukan: {db_path}"
        )

print("Versi DuckDB:", duckdb.__version__)

# ==================================================
# 2. Hubungkan DuckDB dan ekstensi DuckLake
# ==================================================
con = duckdb.connect()

con.execute("INSTALL ducklake")
con.execute("LOAD ducklake")

# Empat bebek lokal dihubungkan sebagai sumber read-only
sources = [
    ("src_soeratin", required_databases[0]),
    ("src_liga1", required_databases[1]),
    ("src_aff", required_databases[2]),
    ("src_fasilitas", required_databases[3]),
]

for alias, db_path in sources:
    con.execute(
        f"ATTACH '{db_path.as_posix()}' "
        f"AS {alias} (READ_ONLY)"
    )
    print("Terhubung:", alias)

# ==================================================
# 3. Buat atau buka DuckLake
# ==================================================
con.execute(
    f"ATTACH 'ducklake:{CATALOG_PATH.as_posix()}' "
    f"AS lake "
    f"(DATA_PATH '{DATA_PATH.as_posix()}/')"
)

print("\nDuckLake berhasil dihubungkan.")
print("Katalog:", CATALOG_PATH)
print("Penyimpanan:", DATA_PATH)

Versi DuckDB: 1.3.2
Terhubung: src_soeratin
Terhubung: src_liga1
Terhubung: src_aff
Terhubung: src_fasilitas

DuckLake berhasil dihubungkan.
Katalog: /content/analisis-pola-lapangan-sepak-bola-jatim/database/ducklake/shared_catalog.ducklake
Penyimpanan: /content/analisis-pola-lapangan-sepak-bola-jatim/database/ducklake/lakehouse_storage


In [11]:
# ==================================================
# 1. Buat schema DuckLake
# ==================================================
schemas = [
    "soeratin",
    "liga1",
    "aff",
    "fasilitas",
    "analytics",
]

for schema in schemas:
    con.execute(
        f"CREATE SCHEMA IF NOT EXISTS lake.{schema}"
    )

# ==================================================
# 2. Salin tabel dari 4 DuckDB lokal
# ==================================================
source_tables = [
    (
        "soeratin", "event_soeratin",
        "src_soeratin", "event_soeratin"
    ),
    (
        "liga1", "event_liga1",
        "src_liga1", "event_liga1"
    ),
    (
        "aff", "event_aff",
        "src_aff", "event_aff"
    ),
    (
        "fasilitas", "fasilitas_source",
        "src_fasilitas", "fasilitas"
    ),
]

for target_schema, target_table, source_db, source_table in source_tables:
    con.execute(
        f"DROP TABLE IF EXISTS "
        f"lake.{target_schema}.{target_table}"
    )

    con.execute(
        f"""
        CREATE TABLE lake.{target_schema}.{target_table} AS
        SELECT *
        FROM {source_db}.main.{source_table}
        """
    )

    count = con.execute(
        f"""
        SELECT COUNT(*)
        FROM lake.{target_schema}.{target_table}
        """
    ).fetchone()[0]

    print(
        f"{target_schema}.{target_table}: {count} baris"
    )

# ==================================================
# 3. Muat dim_venue hasil standardisasi
# ==================================================
dim_path = PROCESSED_DIR / "dim_venue.csv"

if not dim_path.exists():
    raise FileNotFoundError(
        f"File dimensi venue tidak ditemukan: {dim_path}"
    )

df_dim = pd.read_csv(
    dim_path,
    sep=";",
    encoding="utf-8-sig"
)

con.register("tmp_dim_venue", df_dim)

con.execute(
    "DROP TABLE IF EXISTS lake.fasilitas.dim_venue"
)

con.execute("""
    CREATE TABLE lake.fasilitas.dim_venue AS
    SELECT * FROM tmp_dim_venue
""")

con.unregister("tmp_dim_venue")

# ==================================================
# 4. Muat fact_event hasil integrasi awal
# ==================================================
fact_path = PROCESSED_DIR / "fact_event.csv"

if not fact_path.exists():
    raise FileNotFoundError(
        f"File fact_event tidak ditemukan: {fact_path}"
    )

df_fact = pd.read_csv(
    fact_path,
    sep=";",
    encoding="utf-8-sig"
)

con.register("tmp_fact_event", df_fact)

con.execute(
    "DROP TABLE IF EXISTS lake.analytics.fact_event"
)

con.execute("""
    CREATE TABLE lake.analytics.fact_event AS
    SELECT * FROM tmp_fact_event
""")

con.unregister("tmp_fact_event")

print("\nTabel dimensi dan fakta berhasil dimuat.")

soeratin.event_soeratin: 824 baris
liga1.event_liga1: 34 baris
aff.event_aff: 17 baris
fasilitas.fasilitas_source: 67 baris

Tabel dimensi dan fakta berhasil dimuat.


In [12]:
# ==================================================
# 1. Jumlah pertandingan per venue
# ==================================================
con.execute("""
    CREATE OR REPLACE TABLE lake.analytics.mart_aktivitas_venue AS
    SELECT
        d.venue_id,
        d.nama_venue,
        COUNT(f.id_event) AS jumlah_pertandingan,

        SUM(CASE
            WHEN f.id_event IS NOT NULL
             AND f.kompetisi_sumber = 'Soeratin'
            THEN 1 ELSE 0
        END) AS jumlah_soeratin,

        SUM(CASE
            WHEN f.id_event IS NOT NULL
             AND f.kompetisi_sumber = 'Liga 1'
            THEN 1 ELSE 0
        END) AS jumlah_liga1,

        SUM(CASE
            WHEN f.id_event IS NOT NULL
             AND f.kompetisi_sumber = 'AFF U-19'
            THEN 1 ELSE 0
        END) AS jumlah_aff

    FROM lake.fasilitas.dim_venue AS d
    LEFT JOIN lake.analytics.fact_event AS f
        ON d.venue_id = f.venue_id

    GROUP BY d.venue_id, d.nama_venue
""")

# ==================================================
# 2. Aktivitas per venue dan kompetisi
# ==================================================
con.execute("""
    CREATE OR REPLACE TABLE lake.analytics.mart_kompetisi_venue AS
    SELECT
        f.kompetisi_sumber,
        d.venue_id,
        d.nama_venue,
        COUNT(*) AS jumlah_pertandingan

    FROM lake.analytics.fact_event AS f
    INNER JOIN lake.fasilitas.dim_venue AS d
        ON f.venue_id = d.venue_id

    GROUP BY
        f.kompetisi_sumber,
        d.venue_id,
        d.nama_venue
""")

# ==================================================
# 3. Karakteristik fasilitas dan aktivitas
# ==================================================
con.execute("""
    CREATE OR REPLACE TABLE lake.analytics.mart_karakteristik_venue AS
    SELECT
        d.*,
        COALESCE(a.jumlah_pertandingan, 0)
            AS jumlah_pertandingan,
        COALESCE(a.jumlah_soeratin, 0)
            AS jumlah_soeratin,
        COALESCE(a.jumlah_liga1, 0)
            AS jumlah_liga1,
        COALESCE(a.jumlah_aff, 0)
            AS jumlah_aff

    FROM lake.fasilitas.dim_venue AS d
    LEFT JOIN lake.analytics.mart_aktivitas_venue AS a
        ON d.venue_id = a.venue_id
""")

print("Tiga data mart berhasil dibuat.")

Tiga data mart berhasil dibuat.


In [13]:
tables_to_check = [
    ("soeratin", "event_soeratin"),
    ("liga1", "event_liga1"),
    ("aff", "event_aff"),
    ("fasilitas", "fasilitas_source"),
    ("fasilitas", "dim_venue"),
    ("analytics", "fact_event"),
    ("analytics", "mart_aktivitas_venue"),
    ("analytics", "mart_kompetisi_venue"),
    ("analytics", "mart_karakteristik_venue"),
]

validation = []

for schema, table in tables_to_check:
    count = con.execute(
        f"SELECT COUNT(*) FROM lake.{schema}.{table}"
    ).fetchone()[0]

    validation.append({
        "schema": schema,
        "tabel": table,
        "jumlah_baris": count,
    })

display(pd.DataFrame(validation))

# Validasi jumlah event per kompetisi
print("\nJumlah event berdasarkan sumber:")
display(
    con.execute("""
        SELECT
            kompetisi_sumber,
            COUNT(*) AS jumlah_pertandingan
        FROM lake.analytics.fact_event
        GROUP BY kompetisi_sumber
        ORDER BY kompetisi_sumber
    """).df()
)

# Validasi venue
print("\n10 venue dengan pertandingan terbanyak:")
display(
    con.execute("""
        SELECT *
        FROM lake.analytics.mart_aktivitas_venue
        ORDER BY jumlah_pertandingan DESC
        LIMIT 10
    """).df()
)

print("\nKatalog DuckLake:", CATALOG_PATH)
print("Data Parquet:", DATA_PATH)

,schema,tabel,jumlah_baris
0,soeratin,event_soeratin,824
1,liga1,event_liga1,34
2,aff,event_aff,17
3,fasilitas,fasilitas_source,67
4,fasilitas,dim_venue,67
5,analytics,fact_event,875
6,analytics,mart_aktivitas_venue,67
7,analytics,mart_kompetisi_venue,68
8,analytics,mart_karakteristik_venue,67



Jumlah event berdasarkan sumber:


,kompetisi_sumber,jumlah_pertandingan
0,AFF U-19,17
1,Liga 1,34
2,Soeratin,824



10 venue dengan pertandingan terbanyak:


,venue_id,nama_venue,jumlah_pertandingan,jumlah_soeratin,jumlah_liga1,jumlah_aff
0,VEN-E2D9A8D724,Stadion Canda Bhirawa,48,48.0,0.0,0.0
1,VEN-F6BA36BDED,Stadion Surajaya Lamongan,40,40.0,0.0,0.0
2,VEN-B2C3A40636,Gelora Bung Tomo,40,0.0,34.0,6.0
3,VEN-C6D20D7CE6,Stadion Soeprijadi Blitar,37,37.0,0.0,0.0
4,VEN-D74D01EF38,Jember Sport Garden,36,36.0,0.0,0.0
5,VEN-2F9D87832F,Lapangan Pagu Kediri,30,30.0,0.0,0.0
6,VEN-053690598D,Stadion Jenggolo Sidoarjo,29,29.0,0.0,0.0
7,VEN-DA8D0597E0,Gelora Panataran Blitar,28,28.0,0.0,0.0
8,VEN-C18D952F8E,"Lapangan Gajah Mada, Lamongan",24,24.0,0.0,0.0
9,VEN-239BDBF8FE,Lapangan Ngeronggo Kediri,21,21.0,0.0,0.0



Katalog DuckLake: /content/analisis-pola-lapangan-sepak-bola-jatim/database/ducklake/shared_catalog.ducklake
Data Parquet: /content/analisis-pola-lapangan-sepak-bola-jatim/database/ducklake/lakehouse_storage


In [14]:
# ================================================
# VALIDASI AKHIR DATA WAREHOUSE
# ================================================

checks = {
    "Total fact_event": """
        SELECT COUNT(*)
        FROM lake.analytics.fact_event
    """,
    "Total venue": """
        SELECT COUNT(*)
        FROM lake.fasilitas.dim_venue
    """,
    "Total aktivitas pada data mart": """
        SELECT SUM(jumlah_pertandingan)
        FROM lake.analytics.mart_aktivitas_venue
    """,
    "Event tanpa venue_id": """
        SELECT COUNT(*)
        FROM lake.analytics.fact_event
        WHERE venue_id IS NULL
    """,
    "Event dengan venue_id yang tidak ditemukan": """
        SELECT COUNT(*)
        FROM lake.analytics.fact_event f
        LEFT JOIN lake.fasilitas.dim_venue d
            ON f.venue_id = d.venue_id
        WHERE f.venue_id IS NOT NULL
          AND d.venue_id IS NULL
    """,
    "Venue dengan koordinat lengkap": """
        SELECT COUNT(*)
        FROM lake.fasilitas.dim_venue
        WHERE latitude IS NOT NULL
          AND longitude IS NOT NULL
    """,
    "Koordinat di luar rentang": """
        SELECT COUNT(*)
        FROM lake.fasilitas.dim_venue
        WHERE latitude NOT BETWEEN -90 AND 90
           OR longitude NOT BETWEEN -180 AND 180
    """,
}

for nama, query in checks.items():
    hasil = con.execute(query).fetchone()[0]
    print(f"{nama}: {hasil}")

print("\nDistribusi event per kompetisi:")
display(con.execute("""
    SELECT
        kompetisi_sumber,
        COUNT(*) AS jumlah_pertandingan,
        SUM(CASE
            WHEN venue_id IS NOT NULL THEN 1
            ELSE 0
        END) AS pertandingan_dengan_venue
    FROM lake.analytics.fact_event
    GROUP BY kompetisi_sumber
    ORDER BY kompetisi_sumber
""").df())

print("\nDistribusi venue per kompetisi:")
display(con.execute("""
    SELECT
        kompetisi_sumber,
        COUNT(DISTINCT venue_id) AS jumlah_venue
    FROM lake.analytics.fact_event
    WHERE venue_id IS NOT NULL
    GROUP BY kompetisi_sumber
    ORDER BY kompetisi_sumber
""").df())

Total fact_event: 875
Total venue: 67
Total aktivitas pada data mart: 855
Event tanpa venue_id: 20
Event dengan venue_id yang tidak ditemukan: 0
Venue dengan koordinat lengkap: 67
Koordinat di luar rentang: 0

Distribusi event per kompetisi:


,kompetisi_sumber,jumlah_pertandingan,pertandingan_dengan_venue
0,AFF U-19,17,17.0
1,Liga 1,34,34.0
2,Soeratin,824,804.0



Distribusi venue per kompetisi:


,kompetisi_sumber,jumlah_venue
0,AFF U-19,2
1,Liga 1,1
2,Soeratin,65
